In [0]:
#1. leemos el archivo JSON usando "DataFrameReader" de Spark

# Importamos las librerias que se van a utilizar
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Define el la estructura personName
name_schema = StructType(fields = [
    StructField("forename", StringType(), True),
    StructField("surname", StringType(), True)
])

# Define la estrucutura principal
person_schema = StructType([
    StructField("personId", IntegerType(), False),
    StructField("personName", name_schema)        ##Se utiliza el esquema definido anteriormente
])


# Cargamos el archivo utilizando la estructura definida

person_df = spark.read\
    .schema(person_schema)\
    .json("abfss://bronze@lsdata01.dfs.core.windows.net/person.json")

# Mostramos el resultado
display(person_df)

In [0]:
#Paso 2 - Renombrar, añadir y dar formato a las columnas requeridas
from pyspark.sql.functions import col, concat, current_timestamp, lit

person_with_columns_df = person_df\
    .withColumnRenamed("personId", "person_id")\
    .withColumn("ingestion_date", current_timestamp())\
    .withColumn("enviroment", lit("Produccion"))\
    .withColumn("name", 
                concat(
                    col("personName.forename"),
                    lit(" "),
                    col("personName.surname")
                      )
                )


In [0]:
#Paso 3 - Seleccionar las columnas que se requieren 

person_selected_df = person_with_columns_df.select(col("person_id"), col("name"), col("ingestion_date"), col("enviroment"))

display(person_selected_df)

In [0]:
#Paso 4 - Guardar datos en datalake en formato parket
person_selected_df.write.mode("overwrite").parquet("abfss://silver@lsdata01.dfs.core.windows.net/person")

df = spark.read.parquet("abfss://silver@lsdata01.dfs.core.windows.net/person")
display(df)
